# Utils for fine-tuning

hyperparameters to tweak in a basic MLP:
- the number of hidden layers,
- the number of neurons in each hidden layer,
- the activation function used in each hidden layer and in the output layer.

In general, the ReLU activation function (or one of its variants) is a good default for the hidden layers.

For the output layer, in general
- the sigmoid activation function is for binary classification,
- the softmax activation function for multiclass classification,
- no activation function for regression.

If the MLP overfits the training data, try reducing the number of hidden layers and reducing the number of neurons per hidden layer.

## Keras Tuner

In [2]:
import tensorflow as tf

fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)


In [4]:
import sys
if "google.colab" in sys.modules:
    %pip install -q -U keras_tuner~=1.4.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.3 MB/s eta 0:00:00


## RandomSearch

In [5]:
import keras_tuner as kt

def build_model(hp: kt.HyperParameters):
  n_hidden = hp.Int("n_hidden", min_value=0, max_value=8, default=2)
  n_neurons = hp.Int("n_neurons", min_value=16, max_value=256)
  learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2,
                             sampling="log")
  optimizer = hp.Choice("optimizer", values=["sgd", "adam"])
  if optimizer == "sgd":
      optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
  else:
      optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

  model = tf.keras.Sequential()
  model.add(tf.keras.layers.Flatten())
  for _ in range(n_hidden):
      model.add(tf.keras.layers.Dense(n_neurons, activation="relu"))
  model.add(tf.keras.layers.Dense(10, activation="softmax"))
  model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer,
                  metrics=["accuracy"])
  return model

In [6]:
random_search_tuner = kt.RandomSearch(build_model, objective="val_accuracy",
                                      max_trials=5, overwrite=True,
                                      directory="my_fashion_mnist",
                                      project_name="my_rnd_search", seed=42)
random_search_tuner.search(X_train, y_train, epochs = 10, validation_data=(X_valid, y_valid))

Trial 5 Complete [00h 01m 13s]
val_accuracy: 0.8334000110626221

Best val_accuracy So Far: 0.8492000102996826
Total elapsed time: 00h 06m 10s


In [7]:
top3_models = random_search_tuner.get_best_models(num_models=3)
best_model = top3_models[0]

In [8]:
top3_params = random_search_tuner.get_best_hyperparameters(num_trials=3)
top3_params[0].values  # best hyperparameter values

{'n_hidden': 7,
 'n_neurons': 100,
 'learning_rate': 0.0012482904754698163,
 'optimizer': 'sgd'}

In [11]:
# ask oracle:
best_trial = random_search_tuner.oracle.get_best_trials(num_trials=1)[0]
best_trial.summary()

Trial 1 summary
Hyperparameters:
n_hidden: 7
n_neurons: 100
learning_rate: 0.0012482904754698163
optimizer: sgd
Score: 0.8492000102996826


In [12]:
best_trial.metrics.get_last_value("val_accuracy")

np.float64(0.8492000102996826)

if happy, train best model on full data set, evaluate on test, and deploy to production

In [13]:
best_model.fit(X_train_full, y_train_full, epochs=10)
test_loss, test_accuracy = best_model.evaluate(X_test, y_test)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8791 - loss: 0.3308
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8840 - loss: 0.3165
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8882 - loss: 0.3073
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8906 - loss: 0.2977
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8937 - loss: 0.2886
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8961 - loss: 0.2812
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8981 - loss: 0.2740
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9009 - loss: 0.2678
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9032 - loss: 0.2620
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9054 - loss: 0.2562
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8471 - loss: 0.4485


In [14]:
test_accuracy

0.8468000292778015

to fine-tune data preprocessing hps (hyperparameters), for example batch size, subclass kt.HyperModel with build() and fit()

In [15]:
class MyClassificationHyperModel(kt.HyperModel):
    def build(self, hp):
        return build_model(hp)

    def fit(self, hp, model, X, y, **kwargs):
        if hp.Boolean("normalize"):
            norm_layer = tf.keras.layers.Normalization()
            X = norm_layer(X)
        return model.fit(X, y, **kwargs)

## Hyperband tuner:

In [16]:
hyperband_tuner = kt.Hyperband(
    MyClassificationHyperModel(), objective="val_accuracy", seed=42,
    max_epochs=10, factor=3, hyperband_iterations=2,
    overwrite=True, directory="my_fashion_mnist", project_name="hyperband")

In [17]:
from pathlib import Path


root_logdir = Path(hyperband_tuner.project_dir) / "tensorboard"
tensorboard_cb = tf.keras.callbacks.TensorBoard(root_logdir)
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=2)
hyperband_tuner.search(X_train, y_train, epochs=10,
                       validation_data=(X_valid, y_valid),
                       callbacks=[early_stopping_cb, tensorboard_cb])

Trial 60 Complete [00h 01m 18s]
val_accuracy: 0.8294000029563904

Best val_accuracy So Far: 0.8650000095367432
Total elapsed time: 00h 40m 21s


In [ ]:
import sys

if "google.colab" in sys.modules:  # extra code
    %pip install -q -U tensorboard-plugin-profile


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.2/109.2 kB 8.1 MB/s eta 0:00:00


In [ ]:
%tensorboard --logdir {root_logdir}

## Bayesian tuner

better than Hyperband tuner as it zoom in on the best parameters by fitting a probabilisting model, but it has 2 hyperparameters itself.

In [ ]:
bayesian_opt_tuner = kt.BayesianOptimization(
    MyClassificationHyperModel(), objective="val_accuracy", seed=42,
    max_trials=10, alpha=1e-4, beta=2.6,
    overwrite=True, directory="my_fashion_mnist", project_name="bayesian_opt")
bayesian_opt_tuner.search(X_train, y_train, epochs=10,
                          validation_data=(X_valid, y_valid),
                          callbacks=[early_stopping_cb])

## AutoML

## Optimizers and training tricks

**Core optimizers (practically standard)**:


1. ***Adam / AdamW***

- Great default for MLPs.

- AdamW (Adam with decoupled weight decay) is often better behaved than vanilla Adam for regularization.

- Typical starting hyperparams:

  - lr: 1e-3 (then tune down/up),

  - weight_decay: 1e-4 or 1e-5


2. ***SGD with momentum***

- Often converges to slightly better minima, but needs more careful LR tuning.

- Typical start:

  - lr: 0.1 or 0.01

  - momentum: 0.9

  - optional nesterov=True



3. ***Learning rate schedules***

These are very effective fine-tuners of final performance:

- Step decay – multiply LR by e.g. 0.1 at certain epochs.

- Cosine annealing – smooth decay from initial lr → near 0 over training.

- One-cycle policy – increase LR then decrease it (used often with SGD).

In Keras/PyTorch you often use callbacks/schedulers to implement this.



4. ***Regularization & stabilization tricks***

These are also part of “fine-tuning strategy”:

- Weight decay / L2 regularization – almost always helpful for MLPs.

- Dropout, especially in deeper/wider networks.

- BatchNorm / LayerNorm – helps stabilize deeper MLPs.

- Early stopping:
  - Monitor validation loss.

  - Stop when it hasn’t improved for N epochs.

  - Restore best weights.

### Hyperparameter tuning libraries

Optuna, Keras Tuner, Ray Tune

## Recommendation to tune MLP with Keras:

The “best” practical fine-tuning combo:

1. **Optimizer setup (weights fine-tuning)**

- Use AdamW:

  - lr=1e-3 (then tune between 1e-4 and 3e-3)

  - weight_decay=1e-4

- Add:

  - EarlyStopping on validation loss

  - A learning rate scheduler (cosine decay or ReduceLROnPlateau)

2. **Hyperparameter fine-tuning tool**

Something simple & Keras-native → Keras Tuner.

If you want something more powerful / general → Optuna.

3. **What to tune for an MLP**

Common search space:

- Number of hidden layers: 1–5

- Units per layer: 64–1024

- Activation: ReLU / GELU / LeakyReLU

- Dropout: 0–0.5

- Learning rate: 1e-4–3e-3

- Batch size: 32–512

- Weight decay (L2): 1e-6–1e-3